# NPS Open Climate Data — Pipeline

Run from the repo root. Two paths:

| | Path A | Path B |
|---|---|---|
| **Data** | Real (DAYMET + ERA5 via Earth Engine) | Synthetic demo data |
| **Requires** | Google Earth Engine project | Nothing extra |
| **Time** | ~2–4 hrs for all 63 parks | ~10 seconds |

Run **Step 1** (install), then either **Path A** or **Path B**, then **Steps 3–5**.

## 1. Install dependencies

In [ ]:
import subprocess, sys, os

# Make sure we're running from the repo root
repo_root = os.path.dirname(os.path.abspath("pipeline.ipynb"))
os.chdir(repo_root)
print("Working directory:", os.getcwd())

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".",
     "numpy", "pyarrow", "requests", "--quiet"],
    check=True,
)
print("Done.")

---
## Path A — Real Earth Engine export

Skip to **Path B** if you just want synthetic data.

### A1. Authenticate with Earth Engine

You need a [Google Cloud project with the Earth Engine API enabled](https://developers.google.com/earth-engine/guides/access).
The first time this runs it opens a browser window to authorize access.

In [ ]:
import ee

GCP_PROJECT = "your-project-id"  # <-- replace with your GCP project

ee.Authenticate()  # opens browser on first run; skips if already authenticated
ee.Initialize(project=GCP_PROJECT)
print("Earth Engine initialized.")

### A2. Export climate data

The full 1980–present run for all 63 parks takes **2–4 hours**.
To export a subset, set `SLUGS` to a list of park slugs
(find all slugs in `nps_climate_data/parks.py`).

In [ ]:
import subprocess

# Set to a list to export a subset, e.g. ["yellowstone", "glacier"]
SLUGS = None

cmd = ["python", "scripts/01_export_all_parks.py", "--start", "1980-01-01"]
if SLUGS:
    cmd += ["--slugs"] + SLUGS

subprocess.run(cmd, check=True)
print("Export complete — raw files in data/raw/")

---
## Path B — Synthetic demo data

Generates plausible but synthetic climate series for all 63 parks.
Good for testing the full pipeline before running the real EE export.

In [ ]:
import subprocess
subprocess.run(["python", "scripts/03_generate_demo_data.py"], check=True)
print("Demo data written to data/raw/")

---
## 3. Build site JSON summaries

Aggregates raw daily series into annual/seasonal summaries, runs Mann–Kendall
trend tests and Theil–Sen slopes, and writes per-park JSON files.

In [ ]:
import subprocess
subprocess.run(["python", "scripts/02_build_site_data.py"], check=True)
print("Site JSON written to data/site/")

## 4. Fetch park boundaries

Pulls real park polygons from the USGS PAD-US web service, then fills
any gaps with circle approximations.

In [ ]:
import subprocess

result = subprocess.run(["python", "scripts/06_fetch_padus_boundaries.py"])
if result.returncode != 0:
    print("USGS fetch failed — falling back to circle approximations.")

subprocess.run(["python", "scripts/05_generate_boundaries.py"], check=True)
print("Boundaries written to site/public/data/boundaries/")

## 5. Copy data into the site

Copies JSON summaries into `site/public/data/` and gzip-compresses the
raw CSVs for the site's download links.

In [ ]:
import gzip, shutil
from pathlib import Path

pub = Path("site/public/data")
(pub / "parks").mkdir(parents=True, exist_ok=True)

# JSON summaries
shutil.copy("data/site/parks.json", pub / "parks.json")
for f in Path("data/site/parks").glob("*.json"):
    shutil.copy(f, pub / "parks" / f.name)

# Gzip raw CSVs
raw_dst = pub / "raw"
raw_dst.mkdir(exist_ok=True)
for slug_dir in Path("data/raw").iterdir():
    if not slug_dir.is_dir():
        continue
    out_dir = raw_dst / slug_dir.name
    out_dir.mkdir(exist_ok=True)
    for csv in slug_dir.glob("*.csv"):
        with open(csv, "rb") as f_in, gzip.open(out_dir / (csv.name + ".gz"), "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)

n_json = len(list((pub / "parks").glob("*.json")))
n_csv  = len(list(raw_dst.rglob("*.gz")))
print(f"Copied {n_json} park JSON files and {n_csv} gzipped CSVs.")
print(f"\nNext: cd site && npm install && npm run dev")